In [7]:
import matplotlib.pyplot as plt
import numpy as np
import time
import random
from tabulate import tabulate

# Helper functions for generating different types of arrays
def generate_random_array(size, min_val=0, max_val=None):
    if max_val is None:
        max_val = size - 1
    return [random.randint(min_val, max_val) for _ in range(size)]

def generate_sorted_array(size):
    return list(range(size))

def generate_decreasing_sorted_array(size):
    return list(range(size-1, -1, -1))

def generate_nearly_sorted_array(size, swap_percentage=0.1):
    """Generate an array that's mostly sorted but has some elements swapped"""
    arr = list(range(size))
    num_swaps = int(size * swap_percentage)
    for _ in range(num_swaps):
        i, j = random.sample(range(size), 2)
        arr[i], arr[j] = arr[j], arr[i]
    return arr

def generate_reverse_segment_array(size, segment_size=None):
    """Generate an array with a reversed segment in the middle"""
    if segment_size is None:
        segment_size = size // 4
    arr = list(range(size))
    start = (size - segment_size) // 2
    end = start + segment_size
    arr[start:end] = reversed(arr[start:end])
    return arr

def generate_random_sorted_array(size, min_val=0, max_val=None):
    """Generate random values then sort them"""
    arr = generate_random_array(size, min_val, max_val)
    arr.sort()
    return arr

def time_algorithm(algo, array, runs=3):
    """
    Time the execution of an algorithm on the given array.
    """
    times = []
    for _ in range(runs):
        array_copy = array.copy()
        start = time.perf_counter()
        algo(array_copy)
        end = time.perf_counter()
        times.append(end - start)
    return sum(times) / runs

# Algorithm implementations
def algorithm1(A):
    """Power Set Generation algorithm"""
    N = len(A)
    B = [0] * N
    subset_count = [0]
    
    def compute(A, B, i):
        if i >= N:
            subset_count[0] += 1
            return
        B[i] = 0
        compute(A, B, i+1)
        B[i] = A[i]
        compute(A, B, i+1)
        B[i] = 0
        return
    
    compute(A, B, 0)
    return subset_count[0]

def algorithm3(A):
    """Longest Increasing Subsequence algorithm"""
    if not A:
        return 0
    N = len(A)
    B = [1] * N
    
    for i in range(N):
        for j in range(i):
            if A[j] < A[i]:
                B[i] = max(B[i], B[j] + 1)
    
    return max(B)

def algorithm5(A):
    """Bottom-up Merge Sort algorithm"""
    if not A:
        return []
    N = len(A)
    B = [0] * N
    j = 1
    
    while j <= N-1:
        i = 0
        while i <= N-j:
            p = i
            m = i + j - 1
            r = min(i + 2*j - 1, N-1)
            
            for u in range(m, p-1, -1):
                B[u] = A[u]
            
            for v in range(m+1, r+1):
                B[r - (v - (m+1))] = A[v]
            
            u = p
            v = r
            for w in range(p, r+1):
                if B[v] < B[u]:
                    A[w] = B[v]
                    v -= 1
                else:
                    A[w] = B[u]
                    u += 1
            
            i = i + 2*j
        j = 2*j
    
    return A

def algorithm7(A):
    """Counting Sort algorithm"""
    if not A:
        return []
    N = len(A)
    max_val = max(A) + 1 if A else 1
    B = [0] * max_val
    
    for i in range(N):
        B[A[i]] += 1
    
    i = 0
    for j in range(max_val):
        for k in range(B[j]):
            A[i] = j
            i += 1
    
    return A

def generate_table_and_graph(algorithm, algorithm_name, sizes, complexity_str):
    """Generate a comprehensive table and graph for a specific algorithm"""
    
    test_cases = {
        'Random': generate_random_array,
        'Sorted': generate_sorted_array,
        'Decreasing': generate_decreasing_sorted_array,
        'Nearly Sorted': generate_nearly_sorted_array,
        'Reverse Segment': generate_reverse_segment_array,
        'Random Sorted': generate_random_sorted_array
    }
    
    colors = {
        'Random': 'blue',
        'Sorted': 'green',
        'Decreasing': 'red',
        'Nearly Sorted': 'orange',
        'Reverse Segment': 'purple',
        'Random Sorted': 'cyan'
    }
    
    # Run timing tests
    timing_results = {}
    for case_name, case_func in test_cases.items():
        timing_results[case_name] = []
        
        for size in sizes:
            # Special case for exponential algorithm - limit size
            if 'Power Set' in algorithm_name and size > 20:
                timing_results[case_name].append(float('nan'))
                continue
            
            # Handle counting sort value range limitation
            if 'Counting Sort' in algorithm_name and case_name in ['Random', 'Random Sorted']:
                test_array = case_func(size, max_val=size//10)
            else:
                test_array = case_func(size)
            
            time_taken = time_algorithm(algorithm, test_array)
            timing_results[case_name].append(time_taken)
    
    # Create table data
    table_data = []
    for i, size in enumerate(sizes):
        row = [size]
        
        # Add timing values for each case
        for case_name in test_cases:
            time_val = timing_results[case_name][i]
            if np.isnan(time_val):
                row.append('Infeasible')
            else:
                row.append(f"{time_val:.6f}")
        
        # Add random ratio
        if i > 0 and not np.isnan(timing_results['Random'][i]) and not np.isnan(timing_results['Random'][i-1]) and timing_results['Random'][i-1] != 0:
            ratio = timing_results['Random'][i] / timing_results['Random'][i-1]
            row.append(f"{ratio:.2f}")
        else:
            row.append('-')
            
        table_data.append(row)
    
    # Create headers for table
    headers = ['N']
    headers.extend(test_cases.keys())
    headers.append('Random Ratio')
    
    # Create and plot comprehensive graph
    plt.figure(figsize=(12, 8))
    
    for case_name, times in timing_results.items():
        valid_sizes = []
        valid_times = []
        
        # Filter out NaN values for plotting
        for s, t in zip(sizes, times):
            if not np.isnan(t):
                valid_sizes.append(s)
                valid_times.append(t)
        
        plt.plot(valid_sizes, valid_times, 'o-', label=case_name, 
                 color=colors[case_name], linewidth=2)
    
    plt.xlabel('Size of Array (N)')
    plt.ylabel('Execution Time (seconds)')
    plt.title(f'{algorithm_name} Execution Time vs Array Size\n(Time Complexity: {complexity_str})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Use logarithmic scale for certain algorithms
    if 'Power Set' in algorithm_name:
        plt.yscale('log')
        plt.xscale('log')
    elif 'LIS' in algorithm_name:
        plt.yscale('log')
    
    plt.savefig(f'{algorithm_name.replace(" ", "_")}_comprehensive_graph.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Generate summary text
    summary_text = f"""
Figure {algorithm_name}: Execution Times for {algorithm_name}
    
Time complexity: {complexity_str}

The graph shows the relationship between the input array size (N) and the execution time 
for various input types. For {algorithm_name}, we observe that the execution time growth 
is consistent with the theoretical complexity of {complexity_str}.

The different test cases (Random, Sorted, Decreasing, Nearly Sorted, Reverse Segment, 
and Random Sorted) demonstrate how input characteristics affect performance.
"""
    
    return table_data, headers, summary_text

def generate_all_analyses():
    """Generate comprehensive analysis for all algorithms"""
    
    # Algorithm 1: Power Set Generation - O(N*2^N)
    sizes_alg1 = [5, 8, 10, 12, 14, 16, 18]
    table1, headers1, summary1 = generate_table_and_graph(
        algorithm1, "Algorithm 1: Power Set", sizes_alg1, "O(N·2^N)")
    
    # Algorithm 3: Longest Increasing Subsequence - O(N^2)
    sizes_alg3 = [200, 400, 800, 1600, 3200, 6400]
    table3, headers3, summary3 = generate_table_and_graph(
        algorithm3, "Algorithm 3: LIS", sizes_alg3, "O(N²)")
    
    # Algorithm 5: Bottom-up Merge Sort - O(N log N)
    sizes_alg5 = [1000, 2000, 4000, 8000, 16000, 32000, 64000]
    table5, headers5, summary5 = generate_table_and_graph(
        algorithm5, "Algorithm 5: Merge Sort", sizes_alg5, "O(N log N)")
    
    # Algorithm 7: Counting Sort - O(N)
    sizes_alg7 = [10000, 20000, 40000, 80000, 160000, 320000]
    table7, headers7, summary7 = generate_table_and_graph(
        algorithm7, "Algorithm 7: Counting Sort", sizes_alg7, "O(N)")
    
    # Print comprehensive tables
    print("Algorithm 1: Power Set Generation")
    print(tabulate(table1, headers=headers1, tablefmt="grid"))
    print(summary1)
    print("\n" + "="*80 + "\n")
    
    print("Algorithm 3: Longest Increasing Subsequence")
    print(tabulate(table3, headers=headers3, tablefmt="grid"))
    print(summary3)
    print("\n" + "="*80 + "\n")
    
    print("Algorithm 5: Bottom-up Merge Sort")
    print(tabulate(table5, headers=headers5, tablefmt="grid"))
    print(summary5)
    print("\n" + "="*80 + "\n")
    
    print("Algorithm 7: Counting Sort")
    print(tabulate(table7, headers=headers7, tablefmt="grid"))
    print(summary7)
    
    # Generate combined comparison graph
    plt.figure(figsize=(14, 10))
    
    # Subplot for each algorithm
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # Regenerate graphs for the combined plot
    algorithms = [(algorithm1, "Algorithm 1: Power Set", sizes_alg1, "O(N·2^N)"),
                  (algorithm3, "Algorithm 3: LIS", sizes_alg3, "O(N²)"),
                  (algorithm5, "Algorithm 5: Merge Sort", sizes_alg5, "O(N log N)"),
                  (algorithm7, "Algorithm 7: Counting Sort", sizes_alg7, "O(N)")]
    
    for idx, (algo, name, sizes, complexity) in enumerate(algorithms):
        row = idx // 2
        col = idx % 2
        
        # Run timing tests
        timing_results = {}
        test_cases = {
            'Random': generate_random_array,
            'Sorted': generate_sorted_array,
            'Decreasing': generate_decreasing_sorted_array,
            'Nearly Sorted': generate_nearly_sorted_array,
            'Reverse Segment': generate_reverse_segment_array,
            'Random Sorted': generate_random_sorted_array
        }
        
        colors = {
            'Random': 'blue',
            'Sorted': 'green',
            'Decreasing': 'red',
            'Nearly Sorted': 'orange',
            'Reverse Segment': 'purple',
            'Random Sorted': 'cyan'
        }
        
        for case_name, case_func in test_cases.items():
            timing_results[case_name] = []
            
            for size in sizes:
                if 'Power Set' in name and size > 20:
                    timing_results[case_name].append(float('nan'))
                    continue
                
                if 'Counting Sort' in name and case_name in ['Random', 'Random Sorted']:
                    test_array = case_func(size, max_val=size//10)
                else:
                    test_array = case_func(size)
                
                time_taken = time_algorithm(algo, test_array)
                timing_results[case_name].append(time_taken)
        
        # Plot on subplot
        for case_name, times in timing_results.items():
            valid_sizes = []
            valid_times = []
            
            for s, t in zip(sizes, times):
                if not np.isnan(t):
                    valid_sizes.append(s)
                    valid_times.append(t)
            
            axs[row, col].plot(valid_sizes, valid_times, 'o-', label=case_name, 
                              color=colors[case_name], linewidth=2)
        
        axs[row, col].set_xlabel('Size of Array (N)')
        axs[row, col].set_ylabel('Execution Time (seconds)')
        axs[row, col].set_title(f'{name} ({complexity})')
        axs[row, col].legend()
        axs[row, col].grid(True, alpha=0.3)
        
        if 'Power Set' in name:
            axs[row, col].set_yscale('log')
            axs[row, col].set_xscale('log')
        elif 'LIS' in name:
            axs[row, col].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('All_Algorithms_Comprehensive_Comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("\nAll comprehensive graphs have been generated.")

if __name__ == "__main__":
    random.seed(42)  # For reproducibility
    generate_all_analyses()

Algorithm 1: Power Set Generation
+-----+----------+----------+--------------+-----------------+-------------------+-----------------+----------------+
|   N |   Random |   Sorted |   Decreasing |   Nearly Sorted |   Reverse Segment |   Random Sorted | Random Ratio   |
+=====+==========+==========+==============+=================+===================+=================+================+
|   5 | 1.4e-05  | 1.2e-05  |     1e-05    |        1.2e-05  |          1e-05    |        1.1e-05  | -              |
+-----+----------+----------+--------------+-----------------+-------------------+-----------------+----------------+
|   8 | 9.2e-05  | 8.6e-05  |     7.3e-05  |        8.1e-05  |          7.1e-05  |        7.5e-05  | 6.66           |
+-----+----------+----------+--------------+-----------------+-------------------+-----------------+----------------+
|  10 | 0.000443 | 0.00036  |     0.000334 |        0.000354 |          0.000294 |        0.000348 | 4.80           |
+-----+----------+----

<Figure size 1400x1000 with 0 Axes>